# ⚠️ Instructor Demo — The Notebook Everyone Writes First

> **Do NOT run this notebook.** This is for the instructor to demonstrate the anti-pattern.

This is a realistic example of a 'just get it working' ML notebook.
After the demo, we'll discuss what makes it hard to productionize.

**Discussion questions:**
- Where are the hyperparameters defined?
- How would you rerun this with different data?
- How would a colleague test just the feature engineering step?
- What happens if you need to deploy this to a REST endpoint?

In [0]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

# ---- Load data (hardcoded path) ----
df = spark.table("my_workshop.`00_shared`.telco_churn").toPandas()

# ---- "Feature engineering" ----
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
df = df.drop(columns=['customerID'])

# One-hot encode categoricals (the manual way)
cat_cols = df.select_dtypes('object').columns.tolist()
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

X = df.drop(columns=['Churn'])
y = df['Churn']

# ---- Train/test split ----
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ---- Scale (but only numerics — oops, we already encoded everything) ----
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# ---- Train ----
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

# ---- Evaluate ----
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print(f"F1:      {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

# ---- Save model (to a hardcoded local path) ----
import pickle
with open('/tmp/my_churn_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Model saved to /tmp/my_churn_model.pkl")
# ⚠️  This path disappears when the cluster restarts!

## What's Wrong Here?

| Problem | Impact |
|---|---|
| Hardcoded paths and parameters | Can't rerun with different config |
| `scaler` fitted in notebook scope | Can't reproduce at serving time → train/serve skew |
| Model saved to `/tmp` | Lost on cluster restart |
| No MLflow tracking | Can't compare this run to tomorrow's run |
| No tests | Can't verify feature engineering is correct |
| Everything coupled | Can't reuse the feature engineering in a batch job |

**The fix is not magic** — it's engineering discipline:
1. Move config to a YAML file
2. Move feature logic to a tested Python module
3. Use MLflow to track runs and register models
4. Use Unity Catalog to store models permanently

➡️ That's exactly what we'll see in [03_refactored_pipeline.ipynb]($./03_refactored_pipeline)